In [88]:
print("jupyter working ")

jupyter working 


In [89]:
!uv add numpy pandas openpyxl

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Current directory does not exist


In [90]:
import numpy as np 
import pandas as pd 
import openpyxl as op 



In [91]:
df = pd.read_excel("../data/sample.xlsx", sheet_name="MON")


In [92]:
wb_path = "../data/sample.xlsx"
sheet_name = "MON"
def get_date_from_sheet(wb_path, sheet_name):
    meta = pd.read_excel(wb_path, sheet_name=sheet_name)
    if not meta.empty:
        stacked = meta.stack()
        if not stacked.empty:
            return stacked.iloc[0]
    return None 
date = get_date_from_sheet(wb_path, sheet_name)
day_name = date.strftime("%A")
print(date, day_name)

2023-03-06 00:00:00 Monday


In [ ]:
wb_path = "../data/sample.xlsx"

def get_date_from_sheet(wb_path, sheet_name):
    meta = pd.read_excel(wb_path, sheet_name=sheet_name)
    if not meta.empty:
        stacked = meta.stack()
        if not stacked.empty:
            return stacked.iloc[0]
    return None

# Get all sheet names dynamically
import openpyxl
wb = openpyxl.load_workbook(wb_path, read_only=True)
day_sheets = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']

# Only process sheets that exist in the workbook
valid_sheets = [s for s in day_sheets if s in wb.sheetnames]

# Iterate over all valid sheets
results = {}
for sheet in valid_sheets:
    date = get_date_from_sheet(wb_path, sheet)
    if date:
        day_name = date.strftime("%A")
        results[sheet] = {"date": date, "day": day_name}
        print(f"{sheet}: {date} — {day_name}")

MON: 2023-03-06 00:00:00 — Monday
TUE: 2023-07-03 00:00:00 — Monday
WED: 2023-03-08 00:00:00 — Wednesday
THU: 2023-03-09 00:00:00 — Thursday
FRI: 2023-10-03 00:00:00 — Tuesday
SAT: 2023-11-03 00:00:00 — Friday
SUN: 2023-12-03 00:00:00 — Sunday


In [93]:
def extract_data(wb_path, sheet_name):
    # Read without assuming header position
    df = pd.read_excel(wb_path, sheet_name=sheet_name, header=None, engine='openpyxl')

    # --- Step 1: Find the FIRST real header row (contains 'Docket No') ---
    header_row_idx = df[df.apply(lambda row: row.astype(str).str.strip().eq('Docket No').any(), axis=1)].index[0]

    # --- Step 2: Set that row as the header ---
    df.columns = df.iloc[header_row_idx]
    df = df.iloc[header_row_idx + 1:].reset_index(drop=True)  # data starts after header

    # --- Step 3: Keep only rows where Docket No is numeric (your truth signal) ---
    df = df[pd.to_numeric(df['Docket No'], errors='coerce').notna()]

    # --- Step 4: Final clean up ---
    df = df.reset_index(drop=True)
    df['Docket No'] = df['Docket No'].astype(int)

    return df

df_clean = extract_data(wb_path, sheet_name)
print(df_clean.head())

3 No  Docket No  Reg No      Mobile Gender TIME Service CASH CARD RSL  ...  \
0  1      14125  KEN838  0412038383      M  905     PCL  NaN  NaN   0  ...   
1  2      14126  ENA66S  0416951959      F  915     PCL  NaN  NaN   0  ...   
2  3      14127  EEY61G  0401995714      F  930       X   95  NaN   0  ...   
3  4      14128   CN800  0418882000      F  935       D  NaN   75   0  ...   
4  5      14129  JH2211  0412771876      F  945       X  NaN   95   0  ...   

3  NaN  NaN GA machine BLH machine Wongo NaN NaN GA machine BLH machine Wongo  
0  NOV  1.0        NaN         NaN   NaN NaN  16        NaN         NaN   NaN  
1  NaN  2.0        NaN         NaN   NaN NaN  17        NaN         NaN   NaN  
2  NaN  3.0        NaN         NaN   NaN NaN  18        NaN         NaN   NaN  
3  NaN  4.0        NaN         NaN   NaN NaN  19        NaN         NaN   NaN  
4  NaN  5.0        NaN         NaN   NaN NaN  20        NaN         NaN   NaN  

[5 rows x 44 columns]


In [ ]:


day_sheets = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']

def extract_data(wb_path, sheet_name):
    df = pd.read_excel(wb_path, sheet_name=sheet_name, header=None, engine='openpyxl')
    header_row_idx = df[df.apply(lambda row: row.astype(str).str.strip().eq('Docket No').any(), axis=1)].index[0] 
    # Find the first row that contains 'Docket No' and get its index
    df.columns = df.iloc[header_row_idx]
    df = df.iloc[header_row_idx + 1:].reset_index(drop=True) # what does this do? 
    # It takes all rows starting from the one immediately after the header row (header_row_idx + 1) to the end of the DataFrame, and resets the index to start from 0. This effectively removes the header row from the data and ensures that the index is clean and sequential.
    df = df.iloc[:, 0:14] 
    df = df[pd.to_numeric(df['Docket No'], errors='coerce').notna()]
    df = df.reset_index(drop=True)
    df['Docket No'] = df['Docket No'].astype(int)
    return df


# --- Option 1: Dictionary (recommended) ---
dfs = {}
for sheet in valid_sheets:
    dfs[sheet] = extract_data(wb_path, sheet)

# Access like: dfs['MON'], dfs['TUE']
print(dfs['MON'].head())
print(dfs['TUE'].head())
print(dfs['WED'].head())
print(dfs['THU'].head())
print(dfs['FRI'].head())
print(dfs['SAT'].head())
print(dfs['SUN'].head())



3 No  Docket No  Reg No      Mobile Gender TIME Service CASH CARD RSL  GV  \
0  1      14125  KEN838  0412038383      M  905     PCL  NaN  NaN   0  79   
1  2      14126  ENA66S  0416951959      F  915     PCL  NaN  NaN   0  79   
2  3      14127  EEY61G  0401995714      F  930       X   95  NaN   0   0   
3  4      14128   CN800  0418882000      F  935       D  NaN   75   0   0   
4  5      14129  JH2211  0412771876      F  945       X  NaN   95   0   0   

3 GV CASH GV CARD O/Taker  
0     NaN     NaN  HENDRA  
1     NaN     475  HENDRA  
2     NaN     NaN  HENDRA  
3     NaN     NaN  HENDRA  
4     NaN     NaN  HENDRA  
3 No  Docket No  Reg No      Mobile Gender  TIME Service CASH CARD RSL     GV  \
0  1      14146  BZL35W  0466576221      F   915       D  NaN   55   0      0   
1  2      14147  DAJ33N  0450070773      M   950     PCL  NaN  NaN   0  79.16   
2  3      14148  NCB66C  0419696360      F  1000       X   75  NaN   0      0   
3  4      14149  BXC81W  0407015117      F  1

In [106]:
# add a column for the date (metadata) for this sheet 
df_clean['Date'] = date
print(df_clean.head())

3 No  Docket No  Reg No      Mobile Gender TIME Service CASH CARD RSL  ...  \
0  1      14125  KEN838  0412038383      M  905     PCL  NaN  NaN   0  ...   
1  2      14126  ENA66S  0416951959      F  915     PCL  NaN  NaN   0  ...   
2  3      14127  EEY61G  0401995714      F  930       X   95  NaN   0  ...   
3  4      14128   CN800  0418882000      F  935       D  NaN   75   0  ...   
4  5      14129  JH2211  0412771876      F  945       X  NaN   95   0  ...   

3  NaN GA machine BLH machine Wongo NaN NaN GA machine BLH machine Wongo  \
0  1.0        NaN         NaN   NaN NaN  16        NaN         NaN   NaN   
1  2.0        NaN         NaN   NaN NaN  17        NaN         NaN   NaN   
2  3.0        NaN         NaN   NaN NaN  18        NaN         NaN   NaN   
3  4.0        NaN         NaN   NaN NaN  19        NaN         NaN   NaN   
4  5.0        NaN         NaN   NaN NaN  20        NaN         NaN   NaN   

3       Date  
0 2023-12-03  
1 2023-12-03  
2 2023-12-03  
3 2023-12-03  

In [109]:
def extract_data(wb_path, sheet_name):
    df = pd.read_excel(wb_path, sheet_name=sheet_name, header=None, engine='openpyxl')
    header_row_idx = df[df.apply(lambda row: row.astype(str).str.strip().eq('Docket No').any(), axis=1)].index[0]
    df.columns = df.iloc[header_row_idx]
    df = df.iloc[header_row_idx + 1:].reset_index(drop=True)
    df = df.iloc[:, 0:14]  # ← slice columns here BEFORE filtering rows
    df = df[pd.to_numeric(df['Docket No'], errors='coerce').notna()]
    df = df.reset_index(drop=True)
    df['Docket No'] = df['Docket No'].astype(int)
    return df

wb = openpyxl.load_workbook(wb_path, read_only=True)
valid_sheets = [s for s in day_sheets if s in wb.sheetnames]

dfs = {}
for sheet in valid_sheets:
    df = extract_data(wb_path, sheet)        # already 14 cols
    date = get_date_from_sheet(wb_path, sheet)
    df['Date'] = date
    df['Day'] = date.strftime("%A") if date else None
    dfs[sheet] = df
    print(f"\n--- {sheet} ---")
    print(df.head())

df_all = pd.concat(dfs.values(), ignore_index=True)
print(f"\nTotal rows: {len(df_all)}")
print(df_all.head())


--- MON ---
3 No  Docket No  Reg No      Mobile Gender TIME Service CASH CARD RSL  GV  \
0  1      14125  KEN838  0412038383      M  905     PCL  NaN  NaN   0  79   
1  2      14126  ENA66S  0416951959      F  915     PCL  NaN  NaN   0  79   
2  3      14127  EEY61G  0401995714      F  930       X   95  NaN   0   0   
3  4      14128   CN800  0418882000      F  935       D  NaN   75   0   0   
4  5      14129  JH2211  0412771876      F  945       X  NaN   95   0   0   

3 GV CASH GV CARD O/Taker       Date     Day  
0     NaN     NaN  HENDRA 2023-03-06  Monday  
1     NaN     475  HENDRA 2023-03-06  Monday  
2     NaN     NaN  HENDRA 2023-03-06  Monday  
3     NaN     NaN  HENDRA 2023-03-06  Monday  
4     NaN     NaN  HENDRA 2023-03-06  Monday  

--- TUE ---
3 No  Docket No  Reg No      Mobile Gender  TIME Service CASH CARD RSL     GV  \
0  1      14146  BZL35W  0466576221      F   915       D  NaN   55   0      0   
1  2      14147  DAJ33N  0450070773      M   950     PCL  NaN  NaN 

In [110]:
df_all.to_csv("../data/combined_data.csv", index=False)

In [108]:
#give me an csv output of this
df_clean.to_csv('../data/sample_cleaned.csv', index=False)

In [ ]:
# now for every day of the week, we want to do the same thing and then combine into one big dataframe with a date column for each row.
#w each xlsx will have 7 sheet and we want to loop through each sheet, extract the date, extract the data, and then combine into one big dataframe.
